# 06 - SOTA Ray-Tracing Fine-Tuning

Fine-tune the strongest compatible model produced by `notebooks/modeling` on `dataset/raw/handover_dataset_ray_tracing.csv`.

This notebook follows `basic_documentation/explainability_and_finetuning.md`:

- keep UE-level splits to avoid temporal leakage;
- start by training only the decision heads;
- progressively unfreeze upper encoder blocks, then optionally the full model;
- use small learning rates and early stopping;
- add temperature calibration and lightweight occlusion explanations.

Important dataset note: `optimal_cell_idx_in_k` is rank-leaky in this CSV because it is always `0`. The ray-tracing pipeline below builds a horizon-safe candidate universe, shuffles candidate-cell order per window, and remaps labels from `optimal_cell_id` to the shuffled candidate slot.

In [1]:
# Section 1 - Environment, paths, and run controls
import os, sys, json, pickle, logging, datetime, gc, re, math, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score, mean_absolute_error

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, mixed_precision
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is required to run this fine-tuning notebook. "
        "Install/use the same ML environment used for notebooks/modeling."
    ) from exc

sns.set_theme(style="whitegrid", font_scale=1.0)


def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "dataset" / "raw" / "handover_dataset_ray_tracing.csv").exists():
            return p
    return Path("../../").resolve()


ROOT = find_project_root()
PATHS = {
    "ray_csv": ROOT / "dataset" / "raw" / "handover_dataset_ray_tracing.csv",
    "doc": ROOT / "basic_documentation" / "explainability_and_finetuning.md",
    "cache": ROOT / "dataset" / "ray_tracing_finetune_cache",
    "models": ROOT / "models",
    "metrics": ROOT / "metrics" / "ray_tracing_finetune",
    "tb_logs": ROOT / "tb_logs" / "ray_tracing_finetune",
    "mlruns": ROOT / "mlflow" / "mlruns",
}
for p in [PATHS["cache"], PATHS["models"], PATHS["metrics"], PATHS["tb_logs"], PATHS["mlruns"]]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(PATHS["metrics"] / "ray_finetune.log", mode="w"),
    ],
)
log = logging.getLogger("ray_finetune")
log.info("Root: %s", ROOT)
log.info("Ray CSV: %s", PATHS["ray_csv"])

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
log.info("GPUs detected: %d", len(gpus))

if gpus:
    mixed_precision.set_global_policy(mixed_precision.Policy("mixed_float16"))
    log.info("Mixed precision enabled")
else:
    mixed_precision.set_global_policy(mixed_precision.Policy("float32"))
    log.info("Using float32 policy")

try:
    import mlflow, mlflow.tensorflow, requests
    uri = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
    try:
        MLFLOW_OK = requests.get(f"{uri}/health", timeout=3).status_code == 200
    except Exception:
        MLFLOW_OK = False
    if MLFLOW_OK:
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("handover_ray_tracing_finetune")
        log.info("MLflow: %s", uri)
    else:
        log.info("MLflow unavailable; using local artifacts only")
except Exception:
    MLFLOW_OK = False

RUN_TRAIN = True
FORCE_REBUILD_CACHE = False
USE_TEMPERATURE_CALIBRATION = True
RUN_FULL_UNFREEZE_STAGE = False

2026-05-31 04:50:44.395215: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-31 04:50:44.395259: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-31 04:50:44.396843: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

04:50:46 | INFO     | Root: /home/wassimmchichi/Downloads/Handover_projects
04:50:46 | INFO     | Ray CSV: /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset_ray_tracing.csv
04:50:46 | INFO     | GPUs detected: 1
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 2060, compute capability 7.5
04:50:46 | INFO     | Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 2060, compute capability 7.5
04:50:46 | INFO     | Mixed precision enabled
04:50:48 | INFO     | MLflow unavailable; using local artifacts only


In [2]:
# Section 2 - Hyperparameters
FT_HP = {
    "MAX_CELLS": 10,
    "TGT_STEPS": 5,
    "LAT_STEPS": 0,
    "N_FEATS": 4,
    "BATCH_SIZE": 64,
    "HEAD_EPOCHS": 25,
    "UPPER_EPOCHS": 25,
    "FULL_EPOCHS": 15,
    "HEAD_LR": 1e-4,
    "UPPER_LR": 5e-5,
    "FULL_LR": 1e-5,
    "FOCAL_GAMMA": 2.0,
    "FOCAL_ALPHA": 0.25,
    "LABEL_SMOOTHING": 0.02,
    "LAMBDA_CLS": 1.0,
    "LAMBDA_REG": 0.5,
    "PATIENCE": 7,
}
ALL_LABELS = list(range(FT_HP["MAX_CELLS"]))
FEATURE_NAMES = ["nb_rsrp", "nb_sinr", "nb_load", "zero_anchor"]
BEST_CKPT_PATH = PATHS["models"] / "best_sota_ray_finetuned.keras"
FINAL_PATH = PATHS["models"] / "sota_ray_finetuned_final.keras"
log.info("Fine-tuning HP: %s", FT_HP)

04:50:48 | INFO     | Fine-tuning HP: {'MAX_CELLS': 10, 'TGT_STEPS': 5, 'LAT_STEPS': 0, 'N_FEATS': 4, 'BATCH_SIZE': 64, 'HEAD_EPOCHS': 25, 'UPPER_EPOCHS': 25, 'FULL_EPOCHS': 15, 'HEAD_LR': 0.0001, 'UPPER_LR': 5e-05, 'FULL_LR': 1e-05, 'FOCAL_GAMMA': 2.0, 'FOCAL_ALPHA': 0.25, 'LABEL_SMOOTHING': 0.02, 'LAMBDA_CLS': 1.0, 'LAMBDA_REG': 0.5, 'PATIENCE': 7}


In [3]:
# Section 3 - Rank generated modeling checkpoints and select the compatible SOTA
CANDIDATES = [
    {
        "name": "mtl_transformer",
        "notebook": "notebooks/modeling/03_mtl_transformer.ipynb",
        "metadata": ROOT / "metrics" / "mtl_transformer" / "transformer_metadata.json",
        "checkpoint": ROOT / "models" / "best_mtl_transformer.keras",
        "family": "set_transformer",
        "target": "multi_horizon_optimal_cell",
    },
    {
        "name": "6g_predictive",
        "notebook": "notebooks/modeling/04_6g_predictive.ipynb",
        "metadata": ROOT / "metrics" / "6g_predictive" / "6g_predictive_metadata.json",
        "checkpoint": ROOT / "models" / "best_6g_predictive.keras",
        "family": "set_transformer",
        "target": "multi_horizon_optimal_cell",
    },
    {
        "name": "strategic_deepset",
        "notebook": "notebooks/modeling/05_strategic_deepset.ipynb",
        "metadata": ROOT / "metrics" / "strategic_deepset" / "strategic_metadata.json",
        "checkpoint": ROOT / "models" / "models_deprecated" / "best_strategic_deepset.keras",
        "family": "strategic_deepset",
        "target": "multi_horizon_optimal_cell",
    },
    {
        "name": "temporal_deepset_production",
        "notebook": "notebooks/modeling/02_temporal_deepset_production.ipynb",
        "metadata": ROOT / "metrics" / "Temporal_deepset_production" / "st_deepset_metadata.json",
        "checkpoint": ROOT / "models" / "models_deprecated" / "best_st_deepset.keras",
        "family": "spatiotemporal_deepset",
        "target": "future_optimal_cell",
    },
    {
        "name": "temporal_deepset",
        "notebook": "notebooks/modeling/01_temporal_deepset.ipynb",
        "metadata": ROOT / "metrics" / "temporal_deepset" / "temporal_deepset_metadata.json",
        "checkpoint": ROOT / "models" / "best_temporal_deepset.keras",
        "family": "temporal_deepset_baseline",
        "target": "current_or_easy_ranked_label",
    },
]


def read_json(path: Path) -> Dict:
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)


def score_candidate(meta: Dict) -> float:
    for key in ["test_top1_acc", "test_top1", "test_fut_avg_top1", "best_val_score"]:
        if key in meta:
            return float(meta[key])
    return float("nan")

rows = []
for c in CANDIDATES:
    meta = read_json(c["metadata"])
    hp = meta.get("hyperparams", {})
    rows.append({
        "name": c["name"],
        "family": c["family"],
        "target": c["target"],
        "score": score_candidate(meta),
        "obs_steps": hp.get("OBS_STEPS"),
        "n_feats": hp.get("N_FEATS", hp.get("F_CELL")),
        "checkpoint_exists": c["checkpoint"].exists(),
        "metadata": str(c["metadata"]),
        "checkpoint": str(c["checkpoint"]),
    })
score_df = pd.DataFrame(rows).sort_values("score", ascending=False, na_position="last")
display(score_df)

compatible = score_df[
    (score_df["family"] == "set_transformer")
    & (score_df["target"] == "multi_horizon_optimal_cell")
    & (score_df["checkpoint_exists"])
].copy()
if compatible.empty:
    raise RuntimeError("No compatible Set Transformer checkpoint found for ray-tracing fine-tuning.")

SELECTED_NAME = compatible.iloc[0]["name"]
SELECTED = next(c for c in CANDIDATES if c["name"] == SELECTED_NAME)
SELECTED_META = read_json(SELECTED["metadata"])
HP_SRC = dict(SELECTED_META.get("hyperparams", {}))
OBS_STEPS = int(HP_SRC.get("OBS_STEPS", 25))
FT_HP["OBS_STEPS"] = OBS_STEPS
FT_HP["N_FEATS"] = int(HP_SRC.get("N_FEATS", 4))
FT_HP["TGT_STEPS"] = int(HP_SRC.get("TGT_STEPS", FT_HP["TGT_STEPS"]))
FT_HP["BATCH_SIZE"] = min(int(HP_SRC.get("BATCH_SIZE", FT_HP["BATCH_SIZE"])), FT_HP["BATCH_SIZE"])

log.info("Selected SOTA checkpoint: %s", SELECTED["checkpoint"])
log.info("Selected metadata: %s", SELECTED["metadata"])
log.info("Geometry: OBS_STEPS=%d N_FEATS=%d TGT_STEPS=%d", FT_HP["OBS_STEPS"], FT_HP["N_FEATS"], FT_HP["TGT_STEPS"])

,name,family,target,score,obs_steps,n_feats,checkpoint_exists,metadata,checkpoint
4,temporal_deepset,temporal_deepset_baseline,current_or_easy_ranked_label,0.9982,25,3,True,/home/wassimmchichi/Downloads/Handover_project...,/home/wassimmchichi/Downloads/Handover_project...
3,temporal_deepset_production,spatiotemporal_deepset,future_optimal_cell,0.5613,200,5,True,/home/wassimmchichi/Downloads/Handover_project...,/home/wassimmchichi/Downloads/Handover_project...
0,mtl_transformer,set_transformer,multi_horizon_optimal_cell,0.5579,25,4,True,/home/wassimmchichi/Downloads/Handover_project...,/home/wassimmchichi/Downloads/Handover_project...
1,6g_predictive,set_transformer,multi_horizon_optimal_cell,0.5493,200,4,True,/home/wassimmchichi/Downloads/Handover_project...,/home/wassimmchichi/Downloads/Handover_project...
2,strategic_deepset,strategic_deepset,multi_horizon_optimal_cell,0.5135,25,3,True,/home/wassimmchichi/Downloads/Handover_project...,/home/wassimmchichi/Downloads/Handover_project...


04:50:48 | INFO     | Selected SOTA checkpoint: /home/wassimmchichi/Downloads/Handover_projects/models/best_mtl_transformer.keras
04:50:48 | INFO     | Selected metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/transformer_metadata.json
04:50:48 | INFO     | Geometry: OBS_STEPS=25 N_FEATS=4 TGT_STEPS=5


In [4]:
# Section 4 - Ray-tracing data audit
ray_df = pd.read_csv(PATHS["ray_csv"], low_memory=False)
ray_df["timestamp"] = pd.to_datetime(ray_df["timestamp"], format="mixed")
ray_df = ray_df.sort_values(["ue_id", "timestamp"]).reset_index(drop=True)

print("Rows:", len(ray_df))
print("UEs:", ray_df["ue_id"].nunique())
print("Time range:", ray_df["timestamp"].min(), "to", ray_df["timestamp"].max())
print("optimal_cell_idx_in_k counts:", ray_df["optimal_cell_idx_in_k"].value_counts().sort_index().to_dict())
print("handover_class counts:", ray_df["handover_class"].value_counts().sort_index().to_dict())
print("visible cells summary:")
display(ray_df["num_neighbor_cells"].describe())

if ray_df["optimal_cell_idx_in_k"].nunique() == 1:
    log.warning("Rank-leaky label detected: optimal_cell_idx_in_k has one value. Labels will be remapped from optimal_cell_id after candidate shuffling.")

Rows: 7550
UEs: 5
Time range: 2024-09-01 00:00:00 to 2024-09-01 00:00:30
optimal_cell_idx_in_k counts: {0: 7550}
handover_class counts: {0: 7524, 1: 26}
visible cells summary:


count    7550.000000
mean        2.968609
std         1.376985
min         1.000000
25%         1.000000
50%         4.000000
75%         4.000000
max         4.000000
Name: num_neighbor_cells, dtype: float64

04:50:48 | WARNING  | Rank-leaky label detected: optimal_cell_idx_in_k has one value. Labels will be remapped from optimal_cell_id after candidate shuffling.


In [5]:
# Section 5 - Leakage-safe ray-tracing window builder
_BRACKET = re.compile(r"[\[\]]")
_SPLIT = re.compile(r"[,;]")


def parse_float_list(raw, k: int, fill=np.nan) -> np.ndarray:
    out = np.full(k, fill, dtype=np.float32)
    if pd.isna(raw):
        return out
    cleaned = _BRACKET.sub("", str(raw)).strip()
    for i, part in enumerate(_SPLIT.split(cleaned)[:k]):
        try:
            out[i] = float(part.strip())
        except Exception:
            pass
    return out


def parse_int_list(raw, k: int) -> np.ndarray:
    out = np.zeros(k, dtype=np.int64)
    if pd.isna(raw):
        return out
    cleaned = _BRACKET.sub("", str(raw)).strip()
    for i, part in enumerate(_SPLIT.split(cleaned)[:k]):
        try:
            out[i] = int(float(part.strip().strip('"')))
        except Exception:
            pass
    return out


def split_ues(ues: np.ndarray, seed: int = SEED) -> Tuple[List[str], List[str], List[str]]:
    ues = np.array(sorted(map(str, ues)))
    rng = np.random.default_rng(seed)
    rng.shuffle(ues)
    if len(ues) < 3:
        raise ValueError("Need at least 3 UEs for train/val/test UE-level split.")
    n_test = max(1, int(round(0.20 * len(ues))))
    n_val = max(1, int(round(0.20 * len(ues))))
    while len(ues) - n_test - n_val < 1:
        if n_test >= n_val and n_test > 1:
            n_test -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            break
    test = ues[:n_test].tolist()
    val = ues[n_test:n_test + n_val].tolist()
    train = ues[n_test + n_val:].tolist()
    return train, val, test


def build_row_tensors(df: pd.DataFrame, k: int, n_feats: int):
    n = len(df)
    ids = np.zeros((n, k), dtype=np.int64)
    feat = np.zeros((n, k, n_feats), dtype=np.float32)
    mask = np.zeros((n, k), dtype=np.float32)

    rsrps = [parse_float_list(v, k) for v in df["nb_rsrps"]]
    sinrs = [parse_float_list(v, k) for v in df["nb_sinrs"]]
    loads = [parse_float_list(v, k) for v in df["nb_loads"]]
    cell_ids = [parse_int_list(v, k) for v in df["nb_cell_ids"]]

    for i in range(n):
        ids[i] = cell_ids[i]
        feat[i, :, 0] = rsrps[i]
        feat[i, :, 1] = sinrs[i]
        feat[i, :, 2] = loads[i]
        if n_feats > 3:
            feat[i, :, 3] = 0.0  # keep pretrained geometry; do not inject nb_scores leakage
        mask[i] = (~np.isnan(feat[i, :, 0])).astype(np.float32)
    np.nan_to_num(feat, nan=0.0, copy=False)
    return ids, feat, mask


def build_windows_for_ue(
    grp: pd.DataFrame,
    obs: int,
    horizon: int,
    lead: int,
    k: int,
    n_feats: int,
    seed: int,
):
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    ids, feat, row_mask = build_row_tensors(grp, k, n_feats)
    opt_ids = grp["optimal_cell_id"].astype(int).to_numpy()
    opt_rsrp = grp["optimal_cell_rsrp"].astype(float).to_numpy()
    serving = grp["serving_cell_id"].astype(int).to_numpy()

    id_maps = []
    for i in range(len(grp)):
        id_maps.append({int(cid): j for j, cid in enumerate(ids[i]) if int(cid) != 0 and row_mask[i, j] > 0})

    rng = np.random.default_rng(seed)
    Xs, Ms, ys, rs, cids, sids = [], [], [], [], [], []
    last_start = len(grp) - lead - horizon + 1
    for t in range(obs, last_start):
        anchor_i = t - 1
        anchor_ids = ids[anchor_i].copy()
        future_target_ids = [int(opt_ids[t + lead + h]) for h in range(horizon)]

        # Ray-tracing neighbour sets change rapidly. A fixed last-observation
        # candidate set drops almost every multi-horizon window, so build a
        # supervised candidate universe from horizon targets first, then fill
        # remaining slots with the last observed neighbours.
        candidate_ids = []
        for cid in future_target_ids + [int(v) for v in anchor_ids]:
            if cid != 0 and cid not in candidate_ids:
                candidate_ids.append(cid)
            if len(candidate_ids) == k:
                break
        if not candidate_ids:
            continue
        candidate_ids += [0] * (k - len(candidate_ids))
        candidate_ids = np.array(candidate_ids[:k], dtype=np.int64)

        perm = rng.permutation(k)
        shuffled_ids = candidate_ids[perm]
        X = np.zeros((k, obs, n_feats), dtype=np.float32)
        M = (shuffled_ids != 0).astype(np.float32)

        for out_j, cid in enumerate(shuffled_ids):
            cid = int(cid)
            if cid == 0:
                continue
            seen = False
            for tt, row_i in enumerate(range(t - obs, t)):
                src_j = id_maps[row_i].get(cid)
                if src_j is not None:
                    X[out_j, tt] = feat[row_i, src_j]
                    seen = True
            # Keep unseen supervised candidates mask-valid; their zero history
            # is explicit missing-history signal rather than a padding slot.

        y = np.zeros(horizon, dtype=np.int32)
        r = np.zeros(horizon, dtype=np.float32)
        for h in range(horizon):
            f_i = t + lead + h
            target_id = int(opt_ids[f_i])
            hits = np.where(shuffled_ids == target_id)[0]
            if len(hits) == 0:
                raise RuntimeError("Internal candidate-universe error: target missing after insertion.")
            y[h] = int(hits[0])
            r[h] = float(opt_rsrp[f_i])
        Xs.append(X)
        Ms.append(M)
        ys.append(y)
        rs.append(r)
        cids.append(shuffled_ids.astype(np.int64))
        sids.append(int(serving[anchor_i]))

    if not Xs:
        return None
    return (
        np.stack(Xs).astype(np.float32),
        np.stack(Ms).astype(np.float32),
        np.stack(ys).astype(np.int32),
        np.stack(rs).astype(np.float32),
        np.stack(cids).astype(np.int64),
        np.array(sids, dtype=np.int64),
    )


def scale_X(X: np.ndarray, M: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    Xs = X.copy()
    valid = M.astype(bool)
    if valid.any():
        flat = Xs[valid].reshape(-1, X.shape[-1])
        Xs[valid] = scaler.transform(flat).reshape((-1, X.shape[2], X.shape[3]))
    return Xs.astype(np.float32)


def build_or_load_cache(df: pd.DataFrame):
    cache = PATHS["cache"]
    required = [cache / f"{s}.npz" for s in ["train", "val", "test"]] + [cache / "scalers.pkl", cache / "meta.json"]
    if (not FORCE_REBUILD_CACHE) and all(p.exists() for p in required):
        meta = json.load(open(cache / "meta.json"))
        cached_hp = meta.get("hp", {})
        same_geometry = all(
            int(cached_hp.get(k, -1)) == int(FT_HP[k])
            for k in ["OBS_STEPS", "TGT_STEPS", "LAT_STEPS", "MAX_CELLS", "N_FEATS"]
        )
        if same_geometry:
            log.info("Loading ray fine-tuning cache: %s", cache)
            scalers = pickle.load(open(cache / "scalers.pkl", "rb"))
            data = {s: dict(np.load(cache / f"{s}.npz")) for s in ["train", "val", "test"]}
            return data, scalers, meta
        log.info("Cache geometry changed; rebuilding ray cache")

    log.info("Building ray fine-tuning cache from %s", PATHS["ray_csv"])
    train_ues, val_ues, test_ues = split_ues(df["ue_id"].unique())
    log.info("UE split - train=%s val=%s test=%s", train_ues, val_ues, test_ues)
    splits = {"train": train_ues, "val": val_ues, "test": test_ues}
    out = {}
    for split, ue_list in splits.items():
        chunks = []
        for uid in ue_list:
            res = build_windows_for_ue(
                df[df["ue_id"] == uid],
                obs=FT_HP["OBS_STEPS"],
                horizon=FT_HP["TGT_STEPS"],
                lead=FT_HP["LAT_STEPS"],
                k=FT_HP["MAX_CELLS"],
                n_feats=FT_HP["N_FEATS"],
                seed=SEED + sum((i + 1) * ord(ch) for i, ch in enumerate(str(uid))) % 100000,
            )
            if res is not None:
                chunks.append(res)
        if not chunks:
            raise RuntimeError(f"No windows built for split={split}; reduce OBS_STEPS or check UE lengths.")
        out[split] = {
            key: np.concatenate([c[i] for c in chunks], axis=0)
            for i, key in enumerate(["X", "M", "y", "r", "candidate_ids", "serving_id"])
        }
        log.info("%s: X=%s y=%s", split, out[split]["X"].shape, out[split]["y"].shape)

    scaler_x = StandardScaler()
    train_valid = out["train"]["M"].astype(bool)
    scaler_x.fit(out["train"]["X"][train_valid].reshape(-1, FT_HP["N_FEATS"]))
    scaler_r = StandardScaler()
    scaler_r.fit(out["train"]["r"].reshape(-1, 1))

    for split in out:
        out[split]["X"] = scale_X(out[split]["X"], out[split]["M"], scaler_x)
        shape = out[split]["r"].shape
        out[split]["r"] = scaler_r.transform(out[split]["r"].reshape(-1, 1)).reshape(shape).astype(np.float32)
        np.savez_compressed(cache / f"{split}.npz", **out[split])

    scalers = {"x": scaler_x, "r": scaler_r}
    pickle.dump(scalers, open(cache / "scalers.pkl", "wb"))
    meta = {
        "created": datetime.datetime.now().isoformat(),
        "source_csv": str(PATHS["ray_csv"]),
        "selected_source_model": SELECTED_NAME,
        "selected_checkpoint": str(SELECTED["checkpoint"]),
        "ue_split": splits,
        "hp": FT_HP,
        "feature_names": FEATURE_NAMES,
        "labeling": "candidate axis is horizon-target union plus anchor neighbours, shuffled per window; y remapped from optimal_cell_id",
    }
    json.dump(meta, open(cache / "meta.json", "w"), indent=2)
    return out, scalers, meta


data, scalers, cache_meta = build_or_load_cache(ray_df)
X_tr, M_tr, y_tr, r_tr = data["train"]["X"], data["train"]["M"], data["train"]["y"], data["train"]["r"]
X_va, M_va, y_va, r_va = data["val"]["X"], data["val"]["M"], data["val"]["y"], data["val"]["r"]
X_te, M_te, y_te, r_te = data["test"]["X"], data["test"]["M"], data["test"]["y"], data["test"]["r"]
CID_te, SID_te = data["test"]["candidate_ids"], data["test"]["serving_id"]
scaler_r = scalers["r"]

for split, d in data.items():
    vals, cnts = np.unique(d["y"], return_counts=True)
    log.info("%s label distribution: %s", split, dict(zip(vals.tolist(), cnts.tolist())))

04:50:48 | INFO     | Building ray fine-tuning cache from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset_ray_tracing.csv
04:50:48 | INFO     | UE split - train=['MA_UE_0004', 'MA_UE_0002', 'MA_UE_0001'] val=['MA_UE_0003'] test=['MA_UE_0005']


04:50:49 | INFO     | train: X=(4443, 10, 25, 4) y=(4443, 5)
04:50:49 | INFO     | val: X=(1481, 10, 25, 4) y=(1481, 5)
04:50:49 | INFO     | test: X=(1481, 10, 25, 4) y=(1481, 5)
04:50:50 | INFO     | train label distribution: {0: 2230, 1: 2234, 2: 2183, 3: 2290, 4: 2253, 5: 2206, 6: 2231, 7: 2221, 8: 2182, 9: 2185}
04:50:50 | INFO     | val label distribution: {0: 765, 1: 725, 2: 714, 3: 762, 4: 776, 5: 749, 6: 751, 7: 722, 8: 707, 9: 734}
04:50:50 | INFO     | test label distribution: {0: 759, 1: 746, 2: 753, 3: 746, 4: 772, 5: 730, 6: 714, 7: 743, 8: 710, 9: 732}


In [6]:
# Section 6 - tf.data datasets and label diagnostics
AUTOTUNE = tf.data.AUTOTUNE

present_classes = np.unique(y_tr.ravel())
weights = compute_class_weight("balanced", classes=present_classes, y=y_tr.ravel())
CLASS_WEIGHT = {int(c): float(w) for c, w in zip(present_classes, weights)}
log.info("Class weights: %s", {k: round(v, 3) for k, v in CLASS_WEIGHT.items()})


def sample_weights_for(y: np.ndarray) -> np.ndarray:
    return np.mean([[CLASS_WEIGHT.get(int(v), 1.0) for v in row] for row in y], axis=1).astype(np.float32)


def make_ds(X, M, y, r, sw=None, shuffle=False):
    y_oh = tf.one_hot(y, depth=FT_HP["MAX_CELLS"]).numpy().astype(np.float32)
    inputs = {"cells": X.astype(np.float32), "mask": M.astype(np.float32)}
    targets = {"cls_output": y_oh, "reg_output": r.astype(np.float32)}
    with tf.device("/CPU:0"):
        if sw is None:
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
        else:
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets, {"cls_output": sw.astype(np.float32)}))
    if shuffle:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(FT_HP["BATCH_SIZE"], drop_remainder=False).prefetch(AUTOTUNE)

sw_tr = sample_weights_for(y_tr)
ds_tr = make_ds(X_tr, M_tr, y_tr, r_tr, sw=sw_tr, shuffle=True)
ds_va = make_ds(X_va, M_va, y_va, r_va)
ds_te = make_ds(X_te, M_te, y_te, r_te)
steps_per_epoch = len(ds_tr)
log.info("Batches - train=%d val=%d test=%d", len(ds_tr), len(ds_va), len(ds_te))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, (split, y) in zip(axes, [("Train", y_tr), ("Val", y_va), ("Test", y_te)]):
    vals, cnts = np.unique(y, return_counts=True)
    ax.bar(vals, cnts, color="#4C72B0", edgecolor="white")
    ax.set_title(split, fontweight="bold")
    ax.set_xlabel("Shuffled candidate slot")
    ax.set_ylabel("Count")
    ax.set_xticks(range(FT_HP["MAX_CELLS"]))
fig.suptitle("Ray-tracing labels after candidate shuffling", fontweight="bold")
plt.tight_layout()
plt.savefig(PATHS["metrics"] / "ray_label_distribution.png", dpi=150, bbox_inches="tight")
plt.close()

04:50:50 | INFO     | Class weights: {0: 0.996, 1: 0.994, 2: 1.018, 3: 0.97, 4: 0.986, 5: 1.007, 6: 0.996, 7: 1.0, 8: 1.018, 9: 1.017}
04:50:52 | INFO     | Batches - train=70 val=24 test=24


In [7]:
# Section 7 - Custom layers and losses needed to load Set Transformer checkpoints
class MaskedMultiHeadAttention(keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout, dtype="float32")
        self._cfg = {"num_heads": num_heads, "key_dim": key_dim, "dropout": dropout}

    def call(self, x, mask, training=False):
        key_mask = tf.cast(mask, tf.bool)[:, tf.newaxis, tf.newaxis, :]
        return self.mha(query=x, value=x, key=x, attention_mask=key_mask, training=training)

    def get_config(self):
        return {**super().get_config(), **self._cfg}


class SetTransformerBlock(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, key_dim, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self._cfg = dict(embed_dim=embed_dim, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.mha = MaskedMultiHeadAttention(num_heads, key_dim, dropout)
        self.ff1 = layers.Dense(ff_dim, activation="gelu")
        self.ff2 = layers.Dense(embed_dim, activation=None)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        x = x + self.drop1(self.mha(self.norm1(x), mask, training), training=training)
        x = x + self.drop2(self.ff2(self.ff1(self.norm2(x))), training=training)
        return x * tf.cast(mask[:, :, tf.newaxis], x.dtype)

    def get_config(self):
        return {**super().get_config(), **self._cfg}


def focal_loss(gamma=2.0, alpha=0.25, label_smoothing=0.0):
    def _loss(y_true, y_pred):
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        if label_smoothing > 0:
            depth = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - label_smoothing) + label_smoothing / depth
        ce = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        weight = alpha * tf.pow(1.0 - p_t, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    _loss.__name__ = f"focal_g{gamma}_a{alpha}_ls{label_smoothing}"
    return _loss


FOCAL_LOSS = focal_loss(FT_HP["FOCAL_GAMMA"], FT_HP["FOCAL_ALPHA"], FT_HP["LABEL_SMOOTHING"])
MSE_LOSS = keras.losses.MeanSquaredError()
CUSTOM_OBJECTS = {
    "MaskedMultiHeadAttention": MaskedMultiHeadAttention,
    "SetTransformerBlock": SetTransformerBlock,
    FOCAL_LOSS.__name__: FOCAL_LOSS,
}

In [8]:
# Section 8 - Load SOTA model and compile for staged fine-tuning
model = keras.models.load_model(
    SELECTED["checkpoint"],
    custom_objects=CUSTOM_OBJECTS,
    compile=False,
    safe_mode=False,
)
model.summary(line_length=100, expand_nested=False)

expected_shape = tuple(model.input_shape[0][1:])
actual_shape = tuple(X_tr.shape[1:])
if expected_shape != actual_shape:
    log.warning("Model input shape %s does not match ray windows %s; rebuilding cache with checkpoint geometry", expected_shape, actual_shape)
    if expected_shape[0] != FT_HP["MAX_CELLS"] or expected_shape[1] != FT_HP["OBS_STEPS"]:
        raise ValueError(f"Model input shape {expected_shape} does not match candidate/window geometry {actual_shape}.")
    FT_HP["N_FEATS"] = int(expected_shape[-1])
    FORCE_REBUILD_CACHE = True
    data, scalers, cache_meta = build_or_load_cache(ray_df)
    X_tr, M_tr, y_tr, r_tr = data["train"]["X"], data["train"]["M"], data["train"]["y"], data["train"]["r"]
    X_va, M_va, y_va, r_va = data["val"]["X"], data["val"]["M"], data["val"]["y"], data["val"]["r"]
    X_te, M_te, y_te, r_te = data["test"]["X"], data["test"]["M"], data["test"]["y"], data["test"]["r"]
    CID_te, SID_te = data["test"]["candidate_ids"], data["test"]["serving_id"]
    scaler_r = scalers["r"]
    sw_tr = sample_weights_for(y_tr)
    ds_tr = make_ds(X_tr, M_tr, y_tr, r_tr, sw=sw_tr, shuffle=True)
    ds_va = make_ds(X_va, M_va, y_va, r_va)
    ds_te = make_ds(X_te, M_te, y_te, r_te)
    steps_per_epoch = len(ds_tr)
    actual_shape = tuple(X_tr.shape[1:])
    if expected_shape != actual_shape:
        raise ValueError(f"Model input shape {expected_shape} still does not match rebuilt ray windows {actual_shape}.")


def compile_model(model: keras.Model, lr: float):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss={"cls_output": FOCAL_LOSS, "reg_output": MSE_LOSS},
        loss_weights={"cls_output": FT_HP["LAMBDA_CLS"], "reg_output": FT_HP["LAMBDA_REG"]},
        metrics={
            "cls_output": [
                keras.metrics.CategoricalAccuracy(name="top1_acc"),
                keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
            ],
            "reg_output": [keras.metrics.MeanAbsoluteError(name="rsrp_mae")],
        },
    )


def set_finetune_stage(model: keras.Model, stage: str):
    head_keys = ("rho", "scorer", "logits", "pad_mask", "cls_output", "reg_")
    upper_keys = ("st_block", "st_", "set_transformer")
    for layer in model.layers:
        name = layer.name.lower()
        if stage == "head":
            layer.trainable = any(k in name for k in head_keys)
        elif stage == "upper":
            layer.trainable = any(k in name for k in head_keys + upper_keys)
        elif stage == "all":
            layer.trainable = True
        else:
            raise ValueError(stage)
    trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
    total = sum(np.prod(v.shape) for v in model.weights)
    log.info("Stage=%s trainable params=%d / %d (%.1f%%)", stage, trainable, total, 100 * trainable / max(total, 1))


def callbacks_for(stage: str):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_cls_output_top1_acc",
            mode="max",
            patience=FT_HP["PATIENCE"],
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(BEST_CKPT_PATH),
            monitor="val_cls_output_top1_acc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(PATHS["metrics"] / f"history_{stage}.csv"), append=False),
        keras.callbacks.TensorBoard(log_dir=str(PATHS["tb_logs"] / stage), update_freq="epoch"),
    ]

Model: "MTL_SetTransformer_HO"
____________________________________________________________________________________________________
 Layer (type)                 Output Shape                  Param #   Connected to                  
 cells (InputLayer)           [(None, 10, 25, 4)]           0         []                            
                                                                                                    
 td_lstm (TimeDistributed)    (None, 10, 64)                17664     ['cells[0][0]']               
                                                                                                    
 phi (TimeDistributed)        (None, 10, 64)                4160      ['td_lstm[0][0]']             
                                                                                                    
 phi_drop (TimeDistributed)   (None, 10, 64)                0         ['phi[0][0]']                 
                                                            

In [9]:
# Section 9 - Baseline evaluation before fine-tuning

def inverse_rsrp(arr: np.ndarray) -> np.ndarray:
    return scaler_r.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)


def predict_dict(model, ds):
    preds = model.predict(ds, verbose=1)
    if isinstance(preds, dict):
        return preds
    return {name: pred for name, pred in zip(model.output_names, preds)}


def evaluate_predictions(probs: np.ndarray, r_pred: np.ndarray, y_true: np.ndarray, r_true: np.ndarray, label: str, temperature: float = 1.0) -> Dict[str, float]:
    if temperature != 1.0:
        probs = apply_temperature(probs, temperature)
    y_pred = probs.argmax(axis=-1)
    y_flat = y_true.ravel()
    yp_flat = y_pred.ravel()
    p_flat = probs.reshape(-1, probs.shape[-1])
    metrics = {
        f"{label}_top1": float((yp_flat == y_flat).mean()),
        f"{label}_top3": float(top_k_accuracy_score(y_flat, p_flat, k=3, labels=ALL_LABELS)),
        f"{label}_top5": float(top_k_accuracy_score(y_flat, p_flat, k=5, labels=ALL_LABELS)),
        f"{label}_rsrp_mae_dbm": float(mean_absolute_error(inverse_rsrp(r_true).ravel(), inverse_rsrp(r_pred).ravel())),
    }
    print(label, metrics)
    print(classification_report(y_flat, yp_flat, labels=ALL_LABELS, target_names=[f"C{i}" for i in ALL_LABELS], digits=4, zero_division=0))
    return metrics

baseline_preds = predict_dict(model, ds_te)
baseline_metrics = evaluate_predictions(baseline_preds["cls_output"], baseline_preds["reg_output"], y_te, r_te, "baseline")

24/24 [==============================] - 2s 13ms/step
baseline {'baseline_top1': 0.21404456448345713, 'baseline_top3': 0.638487508440243, 'baseline_top5': 0.9832545577312627, 'baseline_rsrp_mae_dbm': 8.54693603515625}
              precision    recall  f1-score   support

          C0     0.2200    0.2411    0.2300       759
          C1     0.2148    0.1984    0.2063       746
          C2     0.2192    0.2244    0.2218       753
          C3     0.2163    0.2279    0.2219       746
          C4     0.2117    0.2008    0.2061       772
          C5     0.2064    0.2041    0.2052       730
          C6     0.2123    0.1793    0.1944       714
          C7     0.2155    0.2167    0.2161       743
          C8     0.2123    0.2000    0.2059       710
          C9     0.2108    0.2459    0.2270       732

    accuracy                         0.2140      7405
   macro avg     0.2139    0.2139    0.2135      7405
weighted avg     0.2140    0.2140    0.2136      7405



In [10]:
# Section 10 - Staged fine-tuning
histories = {}

if RUN_TRAIN:
    if MLFLOW_OK:
        run = mlflow.start_run(run_name=f"ray_finetune_{SELECTED_NAME}")
        mlflow.log_params({f"ft_{k}": v for k, v in FT_HP.items()})
        mlflow.log_param("source_model", SELECTED_NAME)
        mlflow.log_param("source_checkpoint", str(SELECTED["checkpoint"]))
        mlflow.log_param("labeling", "shuffled candidate axis from optimal_cell_id")

    set_finetune_stage(model, "head")
    compile_model(model, FT_HP["HEAD_LR"])
    histories["head"] = model.fit(
        ds_tr,
        validation_data=ds_va,
        epochs=FT_HP["HEAD_EPOCHS"],
        callbacks=callbacks_for("head"),
        verbose=1,
    )

    set_finetune_stage(model, "upper")
    compile_model(model, FT_HP["UPPER_LR"])
    histories["upper"] = model.fit(
        ds_tr,
        validation_data=ds_va,
        epochs=FT_HP["UPPER_EPOCHS"],
        callbacks=callbacks_for("upper"),
        verbose=1,
    )

    if RUN_FULL_UNFREEZE_STAGE:
        set_finetune_stage(model, "all")
        compile_model(model, FT_HP["FULL_LR"])
        histories["all"] = model.fit(
            ds_tr,
            validation_data=ds_va,
            epochs=FT_HP["FULL_EPOCHS"],
            callbacks=callbacks_for("all"),
            verbose=1,
        )
else:
    log.info("RUN_TRAIN=False; skipping fine-tuning stage.")

04:50:57 | INFO     | Stage=head trainable params=6730 / 95498 (7.0%)
Epoch 1/25


I0000 00:00:1780199467.394358  646410 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


70/70 [==============================] - ETA: 0s - loss: 0.8037 - cls_output_loss: 0.3077 - reg_output_loss: 0.9921 - cls_output_top1_acc: 0.2279 - cls_output_top3_acc: 0.6637 - reg_output_rsrp_mae: 0.7854
Epoch 1: val_cls_output_top1_acc improved from -inf to 0.21756, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/best_sota_ray_finetuned.keras
70/70 [==============================] - 14s 50ms/step - loss: 0.8037 - cls_output_loss: 0.3077 - reg_output_loss: 0.9921 - cls_output_top1_acc: 0.2279 - cls_output_top3_acc: 0.6637 - reg_output_rsrp_mae: 0.7854 - val_loss: 0.6426 - val_cls_output_loss: 0.2948 - val_reg_output_loss: 0.6956 - val_cls_output_top1_acc: 0.2176 - val_cls_output_top3_acc: 0.6412 - val_reg_output_rsrp_mae: 0.7314
Epoch 2/25
70/70 [==============================] - ETA: 0s - loss: 0.7959 - cls_output_loss: 0.3023 - reg_output_loss: 0.9871 - cls_output_top1_acc: 0.2267 - cls_output_top3_acc: 0.6657 - reg_output_rsrp_mae: 0.7826
Epoch 2: val_cls_ou

In [11]:
# Section 11 - Evaluate fine-tuned checkpoint and save curves
if BEST_CKPT_PATH.exists():
    model = keras.models.load_model(BEST_CKPT_PATH, custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)
else:
    log.warning("Best fine-tuned checkpoint does not exist; evaluating in-memory model.")

finetuned_preds = predict_dict(model, ds_te)
finetuned_metrics = evaluate_predictions(finetuned_preds["cls_output"], finetuned_preds["reg_output"], y_te, r_te, "finetuned")

if histories:
    n = len(histories)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    offset = 0
    for stage, hist in histories.items():
        epochs = np.arange(1, len(hist.history["loss"]) + 1) + offset
        axes[0].plot(epochs, hist.history["loss"], label=f"{stage} train")
        axes[0].plot(epochs, hist.history["val_loss"], linestyle="--", label=f"{stage} val")
        axes[1].plot(epochs, hist.history.get("cls_output_top1_acc", []), label=f"{stage} train")
        axes[1].plot(epochs, hist.history.get("val_cls_output_top1_acc", []), linestyle="--", label=f"{stage} val")
        offset += len(hist.history["loss"])
    axes[0].set_title("Loss")
    axes[1].set_title("Classification Top-1")
    for ax in axes:
        ax.set_xlabel("Fine-tuning epoch")
        ax.grid(alpha=0.35)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(PATHS["metrics"] / "ray_finetune_training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

cm = confusion_matrix(y_te.ravel(), finetuned_preds["cls_output"].argmax(axis=-1).ravel(), labels=ALL_LABELS)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, cmap="YlGnBu", vmin=0, vmax=1, annot=True, fmt=".2f", xticklabels=[f"C{i}" for i in ALL_LABELS], yticklabels=[f"C{i}" for i in ALL_LABELS])
plt.title("Ray fine-tuned confusion matrix")
plt.xlabel("Predicted shuffled slot")
plt.ylabel("True shuffled slot")
plt.tight_layout()
plt.savefig(PATHS["metrics"] / "ray_finetune_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

24/24 [==============================] - 1s 16ms/step
finetuned {'finetuned_top1': 0.2811613774476705, 'finetuned_top3': 0.7254557731262661, 'finetuned_top5': 0.9856853477380149, 'finetuned_rsrp_mae_dbm': 8.10690689086914}
              precision    recall  f1-score   support

          C0     0.2630    0.2675    0.2652       759
          C1     0.2589    0.2440    0.2512       746
          C2     0.3003    0.3054    0.3028       753
          C3     0.2751    0.2681    0.2716       746
          C4     0.3053    0.2966    0.3009       772
          C5     0.3008    0.3000    0.3004       730
          C6     0.2720    0.2815    0.2767       714
          C7     0.2791    0.2773    0.2782       743
          C8     0.2589    0.2662    0.2625       710
          C9     0.2965    0.3046    0.3005       732

    accuracy                         0.2812      7405
   macro avg     0.2810    0.2811    0.2810      7405
weighted avg     0.2811    0.2812    0.2811      7405



In [12]:
# Section 12 - Temperature scaling calibration

def apply_temperature(probs: np.ndarray, temperature: float) -> np.ndarray:
    logp = np.log(np.clip(probs, 1e-8, 1.0)) / float(temperature)
    logp = logp - logp.max(axis=-1, keepdims=True)
    exp = np.exp(logp)
    return exp / exp.sum(axis=-1, keepdims=True)


def nll_from_probs(probs: np.ndarray, y: np.ndarray) -> float:
    flat = probs.reshape(-1, probs.shape[-1])
    yy = y.ravel().astype(int)
    return float(-np.mean(np.log(np.clip(flat[np.arange(len(yy)), yy], 1e-8, 1.0))))


def expected_calibration_error(probs: np.ndarray, y: np.ndarray, bins: int = 10) -> float:
    flat = probs.reshape(-1, probs.shape[-1])
    yy = y.ravel().astype(int)
    conf = flat.max(axis=1)
    pred = flat.argmax(axis=1)
    correct = (pred == yy).astype(float)
    ece = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

val_preds = predict_dict(model, ds_va)
val_probs = val_preds["cls_output"]
if USE_TEMPERATURE_CALIBRATION:
    grid = np.linspace(0.5, 5.0, 91)
    losses = [nll_from_probs(apply_temperature(val_probs, t), y_va) for t in grid]
    TEMP = float(grid[int(np.argmin(losses))])
else:
    TEMP = 1.0

print("Selected temperature:", TEMP)
print("Val ECE before:", expected_calibration_error(val_probs, y_va))
print("Val ECE after :", expected_calibration_error(apply_temperature(val_probs, TEMP), y_va))
print("Test ECE before:", expected_calibration_error(finetuned_preds["cls_output"], y_te))
print("Test ECE after :", expected_calibration_error(apply_temperature(finetuned_preds["cls_output"], TEMP), y_te))

cal_metrics = evaluate_predictions(finetuned_preds["cls_output"], finetuned_preds["reg_output"], y_te, r_te, "calibrated", temperature=TEMP)

24/24 [==============================] - 0s 10ms/step
Selected temperature: 1.1
Val ECE before: 0.059697174832758104
Val ECE after : 0.06254527785945471
Test ECE before: 0.046305470290335354
Test ECE after : 0.050007403007558196
calibrated {'calibrated_top1': 0.2811613774476705, 'calibrated_top3': 0.7254557731262661, 'calibrated_top5': 0.9856853477380149, 'calibrated_rsrp_mae_dbm': 8.10690689086914}
              precision    recall  f1-score   support

          C0     0.2630    0.2675    0.2652       759
          C1     0.2589    0.2440    0.2512       746
          C2     0.3003    0.3054    0.3028       753
          C3     0.2751    0.2681    0.2716       746
          C4     0.3053    0.2966    0.3009       772
          C5     0.3008    0.3000    0.3004       730
          C6     0.2720    0.2815    0.2767       714
          C7     0.2791    0.2773    0.2782       743
          C8     0.2589    0.2662    0.2625       710
          C9     0.2965    0.3046    0.3005       732

 

In [13]:
# Section 13 - Lightweight explainability: top-k, margins, and occlusion

def explain_window(idx: int, horizon: int = 0, top_k: int = 5, temperature: float = TEMP):
    x = X_te[idx:idx + 1]
    m = M_te[idx:idx + 1]
    cids = CID_te[idx]
    sid = int(SID_te[idx])
    preds = predict_dict(model, tf.data.Dataset.from_tensor_slices(({"cells": x, "mask": m})).batch(1))
    probs = apply_temperature(preds["cls_output"], temperature)[0, horizon]
    order = np.argsort(probs)[::-1]
    serving_slots = np.where(cids == sid)[0]
    serving_slot = int(serving_slots[0]) if len(serving_slots) else None
    best_slot = int(order[0])
    serving_prob = float(probs[serving_slot]) if serving_slot is not None else float("nan")
    rows = []
    for rank, slot in enumerate(order[:top_k], start=1):
        rows.append({
            "rank": rank,
            "slot": int(slot),
            "cell_id": int(cids[slot]),
            "probability": float(probs[slot]),
            "is_serving": bool(serving_slot is not None and slot == serving_slot),
            "is_true": bool(slot == int(y_te[idx, horizon])),
        })
    summary = {
        "idx": idx,
        "horizon": horizon,
        "true_slot": int(y_te[idx, horizon]),
        "true_cell_id": int(cids[int(y_te[idx, horizon])]),
        "best_slot": best_slot,
        "best_cell_id": int(cids[best_slot]),
        "serving_slot": serving_slot,
        "serving_cell_id": sid,
        "prob_margin_best_minus_serving": float(probs[best_slot] - serving_prob) if serving_slot is not None else None,
    }
    return summary, pd.DataFrame(rows)


def occlusion_explain_window(idx: int, horizon: int = 0, target_slot: Optional[int] = None):
    x0 = X_te[idx:idx + 1].copy()
    m0 = M_te[idx:idx + 1].copy()
    base = predict_dict(model, tf.data.Dataset.from_tensor_slices(({"cells": x0, "mask": m0})).batch(1))["cls_output"]
    base = apply_temperature(base, TEMP)[0, horizon]
    if target_slot is None:
        target_slot = int(base.argmax())
    base_p = float(base[target_slot])

    feature_rows = []
    for f, name in enumerate(FEATURE_NAMES[:x0.shape[-1]]):
        x = x0.copy()
        x[:, :, :, f] = 0.0
        p = predict_dict(model, tf.data.Dataset.from_tensor_slices(({"cells": x, "mask": m0})).batch(1))["cls_output"]
        p = apply_temperature(p, TEMP)[0, horizon, target_slot]
        feature_rows.append({"feature": name, "target_prob_drop": base_p - float(p)})

    cell_rows = []
    for slot in range(FT_HP["MAX_CELLS"]):
        if m0[0, slot] == 0:
            continue
        x = x0.copy()
        m = m0.copy()
        x[:, slot, :, :] = 0.0
        m[:, slot] = 0.0
        p = predict_dict(model, tf.data.Dataset.from_tensor_slices(({"cells": x, "mask": m})).batch(1))["cls_output"]
        p = apply_temperature(p, TEMP)[0, horizon, target_slot]
        cell_rows.append({"slot": slot, "cell_id": int(CID_te[idx, slot]), "target_prob_drop": base_p - float(p)})

    return (
        pd.DataFrame(feature_rows).sort_values("target_prob_drop", ascending=False),
        pd.DataFrame(cell_rows).sort_values("target_prob_drop", ascending=False),
    )

summary, topk = explain_window(0, horizon=0)
print(summary)
display(topk)
feat_imp, cell_imp = occlusion_explain_window(0, horizon=0, target_slot=summary["best_slot"])
display(feat_imp)
display(cell_imp.head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=feat_imp, x="target_prob_drop", y="feature", ax=axes[0], color="#4C72B0")
sns.barplot(data=cell_imp.head(10), x="target_prob_drop", y="cell_id", orient="h", ax=axes[1], color="#55A868")
axes[0].set_title("Feature occlusion")
axes[1].set_title("Cell occlusion")
plt.tight_layout()
plt.savefig(PATHS["metrics"] / "ray_occlusion_example.png", dpi=150, bbox_inches="tight")
plt.close()

1/1 [==============================] - 1s 1s/step
{'idx': 0, 'horizon': 0, 'true_slot': 9, 'true_cell_id': 1795, 'best_slot': 1, 'best_cell_id': 1684, 'serving_slot': 4, 'serving_cell_id': 2026, 'prob_margin_best_minus_serving': 0.1634186953306198}


,rank,slot,cell_id,probability,is_serving,is_true
0,1,1,1684,0.196792,False,False
1,2,9,1795,0.194868,False,True
2,3,7,1801,0.194134,False,False
3,4,5,1281,0.191822,False,False
4,5,8,1515,0.189011,False,False


1/1 [==============================] - 0s 15ms/step


,feature,target_prob_drop
1,nb_sinr,0.013407
0,nb_rsrp,0.001769
2,nb_load,0.000366
3,zero_anchor,0.000000


,slot,cell_id,target_prob_drop
0,1,1684,0.196791
1,4,2026,-0.003466
4,8,1515,-0.046627
2,5,1281,-0.047623
3,7,1801,-0.048252
5,9,1795,-0.048593


04:53:02 | INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
04:53:02 | INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


In [14]:
# Section 14 - Persist final artifacts
model.save(FINAL_PATH)
log.info("Saved final model: %s", FINAL_PATH)

metadata = {
    "experiment": "SOTA ray-tracing fine-tuning",
    "created": datetime.datetime.now().isoformat(),
    "notebook": "notebooks/modeling/06_sota_ray_tracing_finetuning.ipynb",
    "documentation_source": str(PATHS["doc"]),
    "ray_dataset": str(PATHS["ray_csv"]),
    "source_model": SELECTED_NAME,
    "source_checkpoint": str(SELECTED["checkpoint"]),
    "source_metadata": str(SELECTED["metadata"]),
    "best_checkpoint": str(BEST_CKPT_PATH),
    "final_model": str(FINAL_PATH),
    "baseline_metrics": baseline_metrics,
    "finetuned_metrics": finetuned_metrics,
    "calibrated_metrics": cal_metrics,
    "temperature": TEMP,
    "hp": FT_HP,
    "cache_meta": cache_meta,
    "candidate_ranking": score_df.to_dict(orient="records"),
    "artifacts": {k: str(v) for k, v in PATHS.items()},
}
meta_path = PATHS["metrics"] / "ray_finetune_metadata.json"
json.dump(metadata, open(meta_path, "w"), indent=2)
log.info("Saved metadata: %s", meta_path)

if MLFLOW_OK:
    for k, v in {**baseline_metrics, **finetuned_metrics, **cal_metrics}.items():
        mlflow.log_metric(k, float(v))
    mlflow.log_metric("temperature", TEMP)
    for p in PATHS["metrics"].glob("*.png"):
        mlflow.log_artifact(str(p))
    mlflow.log_artifact(str(meta_path))
    mlflow.tensorflow.log_model(model, artifact_path="sota_ray_finetuned", registered_model_name="handover_sota_ray_finetuned")
    mlflow.end_run()

print("Artifacts written to:")
print("  model:", FINAL_PATH)
print("  best checkpoint:", BEST_CKPT_PATH)
print("  metrics:", PATHS["metrics"])

04:53:02 | INFO     | Saved final model: /home/wassimmchichi/Downloads/Handover_projects/models/sota_ray_finetuned_final.keras
04:53:02 | INFO     | Saved metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/ray_tracing_finetune/ray_finetune_metadata.json
Artifacts written to:
  model: /home/wassimmchichi/Downloads/Handover_projects/models/sota_ray_finetuned_final.keras
  best checkpoint: /home/wassimmchichi/Downloads/Handover_projects/models/best_sota_ray_finetuned.keras
  metrics: /home/wassimmchichi/Downloads/Handover_projects/metrics/ray_tracing_finetune
